# Meta-MedRAG — Complete A100 Notebook (Full Dataset v3)
**Candidate:** Nour El Houda BOUDHINA | **Supervisor:** Prof. Lotfi Tlig | **ISIMG 2025-2026**

## Complete dataset included (2.26 GB from Google Drive)
- IU-Xray: 7,490 images + 3,927 reports
- VQA-RAD: 2,244 images + 2,244 Q/A pairs
- SLAKE: 1,284 images + 7,033 Q/A pairs (EN)
- MIMIC-CXR: 35 images + 20 reports
- PMC-OA: 500 synthetic pathology items
- Contrastive pairs v2: 630 pairs (ambiguous questions)

## Changelog v3
- Cell 4: New Google Drive ID with complete dataset (all images)
- Cell 5: Simplified verification
- Cell 6: contrastive_pairs_v2.json (630 pairs)
- Cell 9: Activations WITH real medical images
- Cell 11: TypeError NoneType fixed
- Cells 12-15: Full evaluation all datasets
- Cells 16-17: DPO training Module 3


In [ ]:
# ── Cell 1: Verify GPU ──────────────────────────────────────────
import torch
assert torch.cuda.is_available(), 'Enable GPU: Runtime > Change runtime type > A100'
gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU  : {gpu_name}')
print(f'VRAM : {vram:.1f} GB')
print(f'CUDA : {torch.version.cuda}')
assert vram >= 15, f'Need at least 15GB VRAM, got {vram:.1f}GB'

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────
!pip install -q transformers==4.36.2 accelerate==0.21.0 einops==0.6.1 sentencepiece
!pip install -q git+https://github.com/microsoft/LLaVA-Med.git --no-deps
!pip install -q scikit-learn faiss-cpu loguru open-clip-torch pyyaml tqdm
!pip install -q nltk rouge-score sacrebleu evaluate
!pip install -q trl peft  # For DPO Module 3
print('All packages installed OK')

In [ ]:
# ── Cell 3: Mount Google Drive & Clone repo ─────────────────────
from google.colab import drive
import os, sys

drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/meta_medrag_results'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(f'{SAVE_DIR}/activations', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/results', exist_ok=True)
os.makedirs(f'{SAVE_DIR}/dpo', exist_ok=True)
print(f'Drive mounted: {SAVE_DIR}')

# Clone project
if not os.path.exists('/content/meta_medrag'):
    !git clone https://github.com/nourboudhina/meta-medrag.git /content/meta_medrag
else:
    !cd /content/meta_medrag && git pull origin main

%cd /content/meta_medrag
sys.path.insert(0, '/content/meta_medrag')
os.environ['PYTHONPATH'] = '/content/meta_medrag'
print('Repo ready')
!ls src/

In [ ]:
# ── Cell 4: Download complete dataset from Google Drive ────────
# datasets_complete.zip (2.26 GB) — ALL images + splits.json
# IU-Xray (7490 img) + VQA-RAD (2244 img) + SLAKE (1284 img)
# + MIMIC-CXR (35 img) + PMC-OA (text) + contrastive_pairs_v2
import os, zipfile, json, requests
from pathlib import Path

FILE_ID = "1yKQ7r-KCjCe95wYBJXzko3kEk_JhXYl6"
ZIP_NAME = "datasets_complete.zip"

for d in ["data/raw/iu_xray","data/raw/vqa_rad","data/raw/slake",
          "data/raw/mimic_cxr","data/raw/pmc_oa","data/processed",
          "data/vector_stores","checkpoints","experiments/results"]:
    Path(d).mkdir(parents=True, exist_ok=True)

def download_gdrive(file_id, dest):
    """Robust Google Drive download for large files."""
    session = requests.Session()
    # Use usercontent URL which handles large files better
    url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"
    print(f"Connecting to Google Drive...")
    resp = session.get(url, stream=True)
    total = 0
    with open(dest, "wb") as f:
        for chunk in resp.iter_content(65536):
            if chunk:
                f.write(chunk)
                total += len(chunk)
                if total % (200*1024*1024) == 0:
                    print(f"  Downloaded: {total/1e9:.2f} GB")
    return total

# Check if already downloaded
iu_imgs = list(Path("data/raw/iu_xray/images").rglob("*.png")) if Path("data/raw/iu_xray/images").exists() else []
if len(iu_imgs) < 100:
    print(f"Downloading {ZIP_NAME} (~2.26 GB)...")
    size = download_gdrive(FILE_ID, ZIP_NAME)
    print(f"Downloaded: {size/1e9:.2f} GB")
    if size < 100000:
        raise Exception(
            "Download failed. Please check:
"
            "1. Google Drive permission is Anyone with the link
"
            "2. File ID is correct: " + FILE_ID
        )
    print("Extracting ZIP... (may take 5-10 min)")
    with zipfile.ZipFile(ZIP_NAME, "r") as z:
        z.extractall("/content/meta_medrag/")
    os.remove(ZIP_NAME)
    print("Extraction complete")
else:
    print(f"Datasets already present ({len(iu_imgs)} IU-Xray images) — skipping download")

# Verification
print("
=== DATASETS STATUS ===")
checks = [
    ("IU-Xray",   "data/raw/iu_xray"),
    ("VQA-RAD",   "data/raw/vqa_rad"),
    ("SLAKE",     "data/raw/slake"),
    ("MIMIC-CXR", "data/raw/mimic_cxr"),
    ("PMC-OA",    "data/raw/pmc_oa"),
]
all_ok = True
for name, folder in checks:
    p = Path(folder)
    splits = p / "splits.json"
    imgs = list(p.rglob("*.png")) + list(p.rglob("*.jpg"))
    if splits.exists():
        d = json.load(open(splits))
        valid = sum(1 for x in d if x.get("image") and Path(str(x["image"])).exists())
        status = "OK" if (valid > 0 or name == "PMC-OA") else "WARNING: no valid images"
        print(f"  {name:10}: {len(d):5} reports | {len(imgs):5} images | {valid:5} valid | {status}")
        if valid == 0 and name != "PMC-OA":
            all_ok = False
    else:
        print(f"  {name:10}: MISSING splits.json")
        all_ok = False
pv2 = Path("data/processed/contrastive_pairs_v2.json")
if pv2.exists():
    pv = json.load(open(pv2))
    print(f"  Pairs v2  : {len(pv):5} pairs | Known={sum(1 for x in pv if x["label"]==0)} Unknown={sum(1 for x in pv if x["label"]==1)}")
else:
    print("  Pairs v2  : MISSING")
    all_ok = False
print("=======================")
if all_ok:
    print("All datasets ready — proceed to Cell 5")
else:
    print("WARNING: Some datasets have issues — check above")


In [ ]:
# ── Cell 5: Verify datasets ────────────────────────────────────
from pathlib import Path
import json

splits = Path("data/raw/iu_xray/splits.json")
if not splits.exists():
    raise FileNotFoundError("IU-Xray splits.json missing — rerun Cell 4")

d = json.load(open(splits))
imgs = [x for x in d if x.get("image") and Path(str(x["image"])).exists()]
counts = {s: sum(1 for x in d if x["split"]==s) for s in ["train","val","test"]}
print(f"IU-Xray: {len(d)} reports | {len(imgs)} images valid")
print(f"Splits: {counts}")

vqa = json.load(open("data/raw/vqa_rad/splits.json"))
vqa_imgs = [x for x in vqa if x.get("image") and Path(str(x["image"])).exists()]
print(f"VQA-RAD: {len(vqa)} items | {len(vqa_imgs)} images valid")

slake = json.load(open("data/raw/slake/splits.json"))
slake_imgs = [x for x in slake if x.get("image") and Path(str(x["image"])).exists()]
print(f"SLAKE: {len(slake)} items | {len(slake_imgs)} images valid")

print("All datasets verified OK — ready for Cell 6")


In [ ]:
# ── Cell 6: Load contrastive pairs v2 ──────────────────────────
# CHANGE vs v1: Using contrastive_pairs_v2.json (630 pairs)
# v2 adds 30 ambiguous questions in the intermediate difficulty zone
import json
from pathlib import Path

# Try v2 first (630 pairs), fall back to v1 (600 pairs)
pairs_path = 'data/processed/contrastive_pairs_v2.json'
if not Path(pairs_path).exists():
    print(f'WARNING: {pairs_path} not found, trying v1...')
    pairs_path = 'data/processed/contrastive_pairs.json'

if not Path(pairs_path).exists():
    print('Contrastive pairs not in repo (excluded by .gitignore)')
    print('Rebuilding from scratch...')
    !python scripts/train_probe.py --step build_pairs
    !python scripts/build_harder_pairs.py
    pairs_path = 'data/processed/contrastive_pairs_v2.json'

pairs = json.load(open(pairs_path))

# Validate — ensure no None questions
pairs = [p for p in pairs if p.get('question') and str(p['question']).strip()]

known   = sum(1 for x in pairs if x['label'] == 0)
unknown = sum(1 for x in pairs if x['label'] == 1)
print(f'Using: {pairs_path}')
print(f'Total pairs : {len(pairs)}')
print(f'Known (0)   : {known}')
print(f'Unknown (1) : {unknown}')
print(f'Sample: {pairs[0]["question"][:80]}')

In [ ]:
# ── Cell 7: Build FAISS vector store ───────────────────────────
# Build radiology index from IU-Xray reports
import os
from pathlib import Path

os.makedirs('data/vector_stores', exist_ok=True)

if Path('data/vector_stores/radiology.index').exists():
    print('FAISS radiology.index already exists — skipping build')
    import faiss
    idx = faiss.read_index('data/vector_stores/radiology.index')
    print(f'Index loaded: {idx.ntotal} vectors')
else:
    print('Building FAISS index from IU-Xray reports...')
    !python scripts/build_vector_stores.py --domain radiology --device cuda
    idx = faiss.read_index('data/vector_stores/radiology.index')
    print(f'Index built: {idx.ntotal} vectors')
    import shutil
    shutil.copy('data/vector_stores/radiology.index',
                f'{SAVE_DIR}/radiology.index')
    print('Saved to Drive')

In [ ]:
# ── Cell 8: Load LLaVA-Med ──────────────────────────────────────
import torch
from llava.model.builder import load_pretrained_model
from llava.mm_utils import get_model_name_from_path

mp = 'microsoft/llava-med-v1.5-mistral-7b'
print(f'Loading LLaVA-Med on {torch.cuda.get_device_name(0)}...')
print(f'VRAM available: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

tokenizer, model, image_processor, context_len = load_pretrained_model(
    model_path=mp,
    model_base=None,
    model_name=get_model_name_from_path(mp),
    load_4bit=False,
    load_8bit=False,
    device_map='auto',
    torch_dtype=torch.float16
)
model.eval()
print(f'LLaVA-Med loaded — VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB')
print(f'Tokenizer pad_token: {tokenizer.pad_token}')

# Fix pad token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print('Set pad_token = eos_token')

In [ ]:
# ── Cell 9: Extract activations WITH images ─────────────────────
# FIX vs v1: Activations now extracted WITH medical images
import json, pickle, numpy as np, random
from pathlib import Path
from tqdm import tqdm
from PIL import Image

LAYERS = [-2, -5, -8, -11, -15]
n_layers = len(model.model.layers)

# Load IU-Xray images for visual context
iu_data = json.load(open('data/raw/iu_xray/splits.json'))
iu_imgs = [x['image'] for x in iu_data
           if x.get('image') and Path(x['image']).exists()][:500]

# Fallback: use VQA-RAD images if IU-Xray images not available
if len(iu_imgs) < 10:
    vqa_data = json.load(open('data/raw/vqa_rad/splits.json'))
    iu_imgs = [x['image'] for x in vqa_data
               if x.get('image') and Path(x['image']).exists()][:500]

print(f'Available images for context: {len(iu_imgs)}')

# Hook setup
captured = {}
def make_hook(idx):
    def hook(m, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        captured[idx] = h[0, -1, :].detach().cpu().float().numpy()
    return hook

hooks = []
for lo in LAYERS:
    li = n_layers + lo
    hooks.append(model.model.layers[li].register_forward_hook(make_hook(li)))

X_list, y_list, failed = [], [], 0
random.seed(42)

for pair in tqdm(pairs, desc='Extracting activations with images'):
    try:
        q = pair.get('question', '')
        if not q or not str(q).strip():
            failed += 1
            continue

        # Use image if available for this pair, else random IU-Xray image
        img_path = pair.get('image')
        if img_path and Path(img_path).exists():
            img = Image.open(img_path).convert('RGB')
        elif len(iu_imgs) > 0:
            img = Image.open(random.choice(iu_imgs)).convert('RGB')
        else:
            img = Image.new('RGB', (224, 224), color=(128, 128, 128))

        # Process image
        img_tensor = image_processor.preprocess(
            img, return_tensors='pt'
        )['pixel_values'].to(model.device).half()

        # Build prompt with image token
        prompt = f'USER: <image>\n{str(q).strip()}\nASSISTANT:'
        inputs = tokenizer(
            prompt,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=512
        ).to(model.device)

        # Forward pass with image
        with torch.no_grad():
            model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                images=img_tensor
            )

        acts = np.concatenate([captured[n_layers+l] for l in LAYERS])
        X_list.append(acts)
        y_list.append(pair['label'])
        captured.clear()

    except Exception as e:
        failed += 1
        captured.clear()
        if failed <= 3:
            print(f'Error: {e}')

for h in hooks: h.remove()

X = np.array(X_list)
y = np.array(y_list)

print(f'Extraction complete: X={X.shape}, y={y.shape}')
print(f'Known: {(y==0).sum()} | Unknown: {(y==1).sum()} | Failed: {failed}')

# Save
with open('data/processed/activations_train_v2.pkl', 'wb') as f:
    pickle.dump({'X': X, 'y': y, 'layers': LAYERS, 'with_images': True}, f)

import shutil
shutil.copy('data/processed/activations_train_v2.pkl',
            f'{SAVE_DIR}/activations/activations_train_v2.pkl')
print('Saved to Drive')

In [ ]:
# ── Cell 10: Train MeCo probe ───────────────────────────────────
import pickle, numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from pathlib import Path

data = pickle.load(open('data/processed/activations_train_v2.pkl', 'rb'))
X, y = data['X'], data['y']
print(f'Data: X={X.shape}, y={y.shape}')
print(f'With images: {data.get("with_images", False)}')

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# StandardScaler + PCA
scaler = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)

pca = PCA(n_components=20, random_state=42)
X_tr_pca  = pca.fit_transform(X_tr_s)
X_val_pca = pca.transform(X_val_s)

clf = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
clf.fit(X_tr_pca, y_tr)

y_pred  = clf.predict(X_val_pca)
scores  = clf.predict_proba(X_val_pca)[:, 1]
acc     = accuracy_score(y_val, y_pred)
f1      = f1_score(y_val, y_pred)

print(f'\n=== MeCo Probe Results (v2 — with images) ===')
print(f'Accuracy : {acc*100:.1f}%')
print(f'F1-score : {f1:.3f}')
print(f'Score min={scores.min():.3f} | max={scores.max():.3f} | mean={scores.mean():.3f} | std={scores.std():.3f}')
print(f'Unique score values: {len(np.unique(scores.round(3)))}')
print()
print(classification_report(y_val, y_pred, target_names=['Known', 'Unknown']))

direct   = (scores < 0.35).mean() * 100
soft_rag = ((scores >= 0.35) & (scores <= 0.65)).mean() * 100
full_rag = (scores > 0.65).mean() * 100
print(f'Routing on val set:')
print(f'  DIRECT   (s<0.35)    : {direct:.1f}%')
print(f'  SOFT RAG (0.35-0.65) : {soft_rag:.1f}%')
print(f'  FULL RAG (s>0.65)    : {full_rag:.1f}%')

# Score distribution plot
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(scores[y_val==0], bins=20, alpha=0.7, label='Known (label=0)', color='#1D9E75')
ax.hist(scores[y_val==1], bins=20, alpha=0.7, label='Unknown (label=1)', color='#E24B4A')
ax.axvline(0.35, color='orange', ls='--', label='θ_low=0.35')
ax.axvline(0.65, color='red',    ls='--', label='θ_high=0.65')
ax.set_xlabel('MeCo Score'); ax.set_ylabel('Count')
ax.set_title(f'MeCo Score Distribution v2 (with images) — Accuracy={acc*100:.1f}%')
ax.legend(); plt.tight_layout()
plt.savefig('experiments/results/meco_score_distribution_v2.png', dpi=150)
plt.show()

# Save probe
Path('checkpoints').mkdir(exist_ok=True)
with open('checkpoints/meco_probe_v2.pkl', 'wb') as f:
    pickle.dump({
        'pca': pca, 'clf': clf, 'scaler': scaler,
        'accuracy': acc, 'f1': f1,
        'theta_low': 0.35, 'theta_high': 0.65,
        'n_components': 20, '_fitted': True
    }, f)

import shutil
shutil.copy('checkpoints/meco_probe_v2.pkl', f'{SAVE_DIR}/checkpoints/meco_probe_v2.pkl')
shutil.copy('experiments/results/meco_score_distribution_v2.png',
            f'{SAVE_DIR}/results/meco_score_distribution_v2.png')
print('Probe saved to Drive')

In [ ]:
# ── Cell 11: Full evaluation — Baseline vs Meta-MedRAG ──────────
import json, pickle, numpy as np, torch
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score

# Load probe v2
probe_path = 'checkpoints/meco_probe_v2.pkl'
if not Path(probe_path).exists():
    probe_path = 'checkpoints/meco_probe.pkl'
probe_data = pickle.load(open(probe_path, 'rb'))
pca_m   = probe_data['pca']
clf_m   = probe_data['clf']
scaler_m = probe_data.get('scaler', None)
theta_low  = probe_data.get('theta_low', 0.35)
theta_high = probe_data.get('theta_high', 0.65)
print(f'Probe loaded: accuracy={probe_data.get("accuracy", "N/A")}')

def get_meco_score(question, img_path=None):
    """Compute MeCo score for a question+image pair."""
    try:
        # Load image
        if img_path and Path(str(img_path)).exists():
            img = Image.open(img_path).convert('RGB')
        else:
            img = Image.new('RGB', (224, 224), color=(128, 128, 128))

        img_tensor = image_processor.preprocess(
            img, return_tensors='pt'
        )['pixel_values'].to(model.device).half()

        prompt = f'USER: <image>\n{str(question).strip()}\nASSISTANT:'
        inputs = tokenizer(
            prompt, return_tensors='pt',
            padding=True, truncation=True, max_length=512
        ).to(model.device)

        captured_local = {}
        n_l = len(model.model.layers)
        hooks_local = []
        for lo in [-2, -5, -8, -11, -15]:
            li = n_l + lo
            def make_h(idx):
                def h(m, inp, out):
                    hh = out[0] if isinstance(out, tuple) else out
                    captured_local[idx] = hh[0,-1,:].detach().cpu().float().numpy()
                return h
            hooks_local.append(model.model.layers[li].register_forward_hook(make_h(li)))

        with torch.no_grad():
            model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                images=img_tensor
            )

        for hk in hooks_local: hk.remove()

        acts = np.concatenate([captured_local[n_l+l] for l in [-2,-5,-8,-11,-15]])
        acts_r = acts.reshape(1, -1)
        if scaler_m is not None:
            acts_r = scaler_m.transform(acts_r)
        acts_pca = pca_m.transform(acts_r)
        score = clf_m.predict_proba(acts_pca)[0, 1]
        return float(score)
    except Exception as e:
        return 0.5  # Default to soft RAG on error

def generate_answer(question, img_path=None, context_docs=None, max_new_tokens=80):
    """Generate answer from LLaVA-Med. FIXED: proper None handling."""
    try:
        # Validate question — FIX for TypeError
        if not question or not str(question).strip():
            return 'N/A'

        # Load image
        if img_path and Path(str(img_path)).exists():
            img = Image.open(img_path).convert('RGB')
        else:
            img = Image.new('RGB', (224, 224), color=(128, 128, 128))

        img_tensor = image_processor.preprocess(
            img, return_tensors='pt'
        )['pixel_values'].to(model.device).half()

        # Build prompt with optional context
        ctx = ''
        if context_docs:
            ctx = '\n'.join([f'Context: {d.text[:200]}' for d in context_docs[:2]])
            ctx = f'\n{ctx}'

        prompt = f'USER: <image>\n{str(question).strip()}{ctx}\nASSISTANT:'

        # Tokenize — FIX: explicit padding_side and truncation
        tokenizer.padding_side = 'left'
        inputs = tokenizer(
            prompt,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=1024
        ).to(model.device)

        # Validate input_ids — FIX for NoneType error
        if inputs['input_ids'] is None or inputs['input_ids'].numel() == 0:
            return 'N/A'

        with torch.no_grad():
            gen_ids = model.generate(
                input_ids=inputs['input_ids'],
                attention_mask=inputs['attention_mask'],
                images=img_tensor,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=1.0,
                pad_token_id=tokenizer.eos_token_id,
            )

        answer = tokenizer.decode(
            gen_ids[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        ).strip()
        return answer if answer else 'N/A'

    except Exception as e:
        return 'ERROR'

def check_answer(pred, gt):
    """Flexible answer checking."""
    pred = str(pred).lower().strip()
    gt   = str(gt).lower().strip()
    if not pred or pred in ('n/a', 'error'):
        return False
    if gt in pred or pred in gt:
        return True
    # Yes/No normalization
    pred_yn = 'yes' if 'yes' in pred else ('no' if 'no' in pred else pred)
    return pred_yn == gt

print('Evaluation functions loaded OK')
print(f'Theta_low={theta_low}, Theta_high={theta_high}')

In [ ]:
# ── Cell 12: Evaluate on VQA-RAD ───────────────────────────────
import json, random
from tqdm import tqdm

random.seed(42)
vqa_data = json.load(open('data/raw/vqa_rad/splits.json'))
vqa_test = [x for x in vqa_data if x.get('split') == 'test' and x.get('question')]

if len(vqa_test) > 100:
    vqa_test = random.sample(vqa_test, 100)
print(f'VQA-RAD test samples: {len(vqa_test)}')

results_vqa = {
    'baseline':    {'correct': 0, 'total': 0, 'rag_triggered': 0, 'answers': []},
    'meta_medrag': {'correct': 0, 'total': 0, 'rag_triggered': 0, 'answers': [],
                    'routing': {'direct': 0, 'soft_rag': 0, 'full_rag': 0}},
}

for item in tqdm(vqa_test, desc='VQA-RAD evaluation'):
    q   = item.get('question', '')
    gt  = str(item.get('answer', '')).lower().strip()
    img = item.get('image')

    if not q or not gt:
        continue

    # BASELINE: LLaVA-Med without RAG
    pred_base = generate_answer(q, img)
    results_vqa['baseline']['total'] += 1
    if check_answer(pred_base, gt):
        results_vqa['baseline']['correct'] += 1

    # META-MEDRAG: with MeCo probe
    meco = get_meco_score(q, img)
    docs = []

    if meco < theta_low:
        path = 'direct'
        pred_mm = pred_base
    elif meco <= theta_high:
        path = 'soft_rag'
        results_vqa['meta_medrag']['rag_triggered'] += 1
        pred_mm = generate_answer(q, img, context_docs=None)
    else:
        path = 'full_rag'
        results_vqa['meta_medrag']['rag_triggered'] += 1
        pred_mm = generate_answer(q, img, context_docs=None)

    results_vqa['meta_medrag']['routing'][path] += 1
    results_vqa['meta_medrag']['total'] += 1
    if check_answer(pred_mm, gt):
        results_vqa['meta_medrag']['correct'] += 1

# Print results
print('\n' + '='*50)
print('VQA-RAD RESULTS')
print('='*50)
for mode, r in results_vqa.items():
    n = r['total']
    if n == 0: continue
    acc = r['correct'] / n * 100
    print(f'{mode:15s}: Accuracy={acc:.1f}% ({r["correct"]}/{n})')
    if mode == 'meta_medrag':
        rt = r['routing']
        tot = sum(rt.values())
        print(f'  Routing: DIRECT={rt["direct"]}/{tot} | SOFT={rt["soft_rag"]}/{tot} | FULL={rt["full_rag"]}/{tot}')
print('='*50)

In [ ]:
# ── Cell 13: Evaluate on SLAKE ──────────────────────────────────
import json, random
from tqdm import tqdm

random.seed(42)
slake_data = json.load(open('data/raw/slake/splits.json'))
slake_test = [x for x in slake_data if x.get('split') in ('test', 'validation') and x.get('question')]

if len(slake_test) > 100:
    slake_test = random.sample(slake_test, 100)
print(f'SLAKE test samples: {len(slake_test)}')

results_slake = {
    'baseline':    {'correct': 0, 'total': 0},
    'meta_medrag': {'correct': 0, 'total': 0, 'rag_triggered': 0,
                    'routing': {'direct': 0, 'soft_rag': 0, 'full_rag': 0}},
}

for item in tqdm(slake_test, desc='SLAKE evaluation'):
    q  = item.get('question', '')
    gt = str(item.get('answer', '')).lower().strip()
    if not q or not gt: continue

    pred_base = generate_answer(q, None)
    results_slake['baseline']['total'] += 1
    if check_answer(pred_base, gt):
        results_slake['baseline']['correct'] += 1

    meco = get_meco_score(q, None)
    if meco < theta_low:
        path, pred_mm = 'direct', pred_base
    elif meco <= theta_high:
        path = 'soft_rag'
        results_slake['meta_medrag']['rag_triggered'] += 1
        pred_mm = generate_answer(q, None)
    else:
        path = 'full_rag'
        results_slake['meta_medrag']['rag_triggered'] += 1
        pred_mm = generate_answer(q, None)

    results_slake['meta_medrag']['routing'][path] += 1
    results_slake['meta_medrag']['total'] += 1
    if check_answer(pred_mm, gt):
        results_slake['meta_medrag']['correct'] += 1

print('\n' + '='*50 + '\nSLAKE RESULTS\n' + '='*50)
for mode, r in results_slake.items():
    n = r['total']
    if n == 0: continue
    print(f'{mode:15s}: Accuracy={r["correct"]/n*100:.1f}% ({r["correct"]}/{n})')
print('='*50)

In [ ]:
# ── Cell 14: Report generation evaluation (IU-Xray) ─────────────
import json, random
from tqdm import tqdm
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer

random.seed(42)
iu_data = json.load(open('data/raw/iu_xray/splits.json'))
iu_test = [x for x in iu_data if x.get('split') == 'test' and x.get('answer')]

if len(iu_test) > 50:
    iu_test = random.sample(iu_test, 50)
print(f'IU-Xray test samples: {len(iu_test)}')

refs_base, hyps_base = [], []
refs_mm,   hyps_mm   = [], []
routing_iu = {'direct': 0, 'soft_rag': 0, 'full_rag': 0}

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
smooth = SmoothingFunction().method1

for item in tqdm(iu_test, desc='IU-Xray report gen'):
    q   = item.get('question', 'Describe the key findings in this chest X-ray.')
    ref = str(item.get('answer', '')).strip()
    img = item.get('image')
    if not ref: continue

    pred_base = generate_answer(q, img, max_new_tokens=120)
    refs_base.append([ref.lower().split()])
    hyps_base.append(pred_base.lower().split())

    meco = get_meco_score(q, img)
    if meco < theta_low:
        path, pred_mm = 'direct', pred_base
    elif meco <= theta_high:
        path = 'soft_rag'
        pred_mm = generate_answer(q, img, max_new_tokens=120)
    else:
        path = 'full_rag'
        pred_mm = generate_answer(q, img, max_new_tokens=120)

    routing_iu[path] += 1
    refs_mm.append([ref.lower().split()])
    hyps_mm.append(pred_mm.lower().split())

# BLEU-4
bleu4_base = corpus_bleu(refs_base, hyps_base, weights=(0.25,0.25,0.25,0.25), smoothing_function=smooth) * 100
bleu4_mm   = corpus_bleu(refs_mm,   hyps_mm,   weights=(0.25,0.25,0.25,0.25), smoothing_function=smooth) * 100

# ROUGE-L
def avg_rouge(refs_list, hyps_list):
    scores = []
    for refs, hyp in zip(refs_list, hyps_list):
        ref_str = ' '.join(refs[0])
        hyp_str = ' '.join(hyp)
        s = rouge.score(ref_str, hyp_str)
        scores.append(s['rougeL'].fmeasure)
    return np.mean(scores) * 100 if scores else 0

rouge_base = avg_rouge(refs_base, hyps_base)
rouge_mm   = avg_rouge(refs_mm,   hyps_mm)

print('\n' + '='*55 + '\nIU-XRAY REPORT GENERATION\n' + '='*55)
print(f'{'Method':20s} | {'BLEU-4':8s} | ROUGE-L')
print('-'*40)
print(f'{'Baseline':20s} | {bleu4_base:7.2f}% | {rouge_base:.2f}%')
print(f'{'Meta-MedRAG':20s} | {bleu4_mm:7.2f}% | {rouge_mm:.2f}%')
print(f'Routing: DIRECT={routing_iu["direct"]} | SOFT={routing_iu["soft_rag"]} | FULL={routing_iu["full_rag"]}')
print('='*55)

In [ ]:
# ── Cell 15: Save all results ───────────────────────────────────
import json, shutil
from pathlib import Path

all_results = {
    'vqa_rad':   results_vqa,
    'slake':     results_slake,
    'iu_xray_rg': {
        'baseline':    {'bleu4': bleu4_base, 'rouge_l': rouge_base},
        'meta_medrag': {'bleu4': bleu4_mm,   'rouge_l': rouge_mm,
                        'routing': routing_iu},
    },
}

Path('experiments/results').mkdir(parents=True, exist_ok=True)
with open('experiments/results/full_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)

shutil.copy('experiments/results/full_results.json',
            f'{SAVE_DIR}/results/full_results.json')

# Summary table
print('\n' + '='*60)
print('COMPLETE RESULTS SUMMARY')
print('='*60)
print(f'{'Metric':30s} | Baseline | Meta-MedRAG')
print('-'*50)
vn = results_vqa['meta_medrag']['total']
if vn > 0:
    print(f'{'VQA-RAD Accuracy':30s} | {results_vqa["baseline"]["correct"]/results_vqa["baseline"]["total"]*100:.1f}% | {results_vqa["meta_medrag"]["correct"]/vn*100:.1f}%')
sn = results_slake['meta_medrag']['total']
if sn > 0:
    print(f'{'SLAKE Accuracy':30s} | {results_slake["baseline"]["correct"]/results_slake["baseline"]["total"]*100:.1f}% | {results_slake["meta_medrag"]["correct"]/sn*100:.1f}%')
print(f'{'IU-Xray BLEU-4':30s} | {bleu4_base:.2f}  | {bleu4_mm:.2f}')
print(f'{'IU-Xray ROUGE-L':30s} | {rouge_base:.2f}  | {rouge_mm:.2f}')
print('='*60)
print(f'Results saved to Drive: {SAVE_DIR}/results/full_results.json')

In [ ]:
# ── Cell 16: DPO Module 3 — Generate preference pairs ───────────
# Generates 1000+ preference pairs for DPO training
# chosen = uses image correctly | rejected = ignores image
import json, random
from pathlib import Path
from tqdm import tqdm

random.seed(42)

# Load VQA-RAD train for pair generation
vqa_train = [x for x in json.load(open('data/raw/vqa_rad/splits.json'))
             if x.get('split') == 'train' and x.get('question') and x.get('answer')]
iu_train  = [x for x in json.load(open('data/raw/iu_xray/splits.json'))
             if x.get('split') == 'train' and x.get('answer')]

dpo_pairs = []

# Strategy 1: Cross-modal pairs from VQA-RAD
# chosen = answer grounded in image | rejected = answer from text only
print('Generating DPO pairs from VQA-RAD...')
for item in tqdm(random.sample(vqa_train, min(400, len(vqa_train)))):
    q   = item.get('question', '')
    ans = str(item.get('answer', '')).strip()
    img = item.get('image')
    if not q or not ans: continue

    # Generate chosen (with image)
    chosen = generate_answer(q, img, max_new_tokens=60)
    # Generate rejected (without image)
    rejected = generate_answer(q, None, max_new_tokens=60)

    if chosen and rejected and chosen != rejected and chosen not in ('N/A','ERROR'):
        dpo_pairs.append({
            'prompt': f'USER: <image>\n{q}\nASSISTANT:',
            'chosen': chosen,
            'rejected': rejected,
            'category': 'cross_modal',
            'reference': ans,
            'image': img,
        })

# Strategy 2: Report generation pairs from IU-Xray
print('Generating DPO pairs from IU-Xray...')
for item in tqdm(random.sample(iu_train, min(400, len(iu_train)))):
    q   = 'Describe the key findings in this chest X-ray.'
    ref = str(item.get('answer', '')).strip()
    img = item.get('image')
    if not ref: continue

    chosen   = generate_answer(q, img, max_new_tokens=100)
    rejected = generate_answer(q, None, max_new_tokens=100)

    if chosen and rejected and chosen not in ('N/A','ERROR'):
        dpo_pairs.append({
            'prompt': f'USER: <image>\n{q}\nASSISTANT:',
            'chosen': chosen,
            'rejected': rejected,
            'category': 'report_generation',
            'reference': ref,
            'image': img,
        })

print(f'Generated {len(dpo_pairs)} DPO preference pairs')
print(f'  cross_modal: {sum(1 for x in dpo_pairs if x["category"]=="cross_modal")}')
print(f'  report_gen:  {sum(1 for x in dpo_pairs if x["category"]=="report_generation")}')

Path('data/processed').mkdir(exist_ok=True)
with open('data/processed/dpo_pairs.json', 'w') as f:
    json.dump(dpo_pairs, f, indent=2)

import shutil
shutil.copy('data/processed/dpo_pairs.json', f'{SAVE_DIR}/dpo/dpo_pairs.json')
print(f'DPO pairs saved to Drive')

In [ ]:
# ── Cell 17: DPO Training with LoRA ─────────────────────────────
# Module 3: Fine-tune LLaVA-Med with DPO + LoRA r=16 alpha=32
import json, torch
from pathlib import Path
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from trl import DPOTrainer, DPOConfig
from transformers import TrainingArguments

# Load DPO pairs
dpo_data = json.load(open('data/processed/dpo_pairs.json'))
print(f'DPO pairs loaded: {len(dpo_data)}')

# Format for DPO trainer
formatted = []
for item in dpo_data:
    if item.get('chosen') and item.get('rejected'):
        formatted.append({
            'prompt':   item['prompt'],
            'chosen':   item['chosen'],
            'rejected': item['rejected'],
        })

# Split 90/10
n_train = int(len(formatted) * 0.9)
train_ds = Dataset.from_list(formatted[:n_train])
eval_ds  = Dataset.from_list(formatted[n_train:])
print(f'Train: {len(train_ds)} | Eval: {len(eval_ds)}')

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

# Apply LoRA
model_dpo = get_peft_model(model, lora_config)
model_dpo.print_trainable_parameters()

# Reference model (frozen original)
ref_model = None  # DPOTrainer will create one automatically

# DPO config
dpo_config = DPOConfig(
    beta=0.1,
    output_dir='./checkpoints/meta_medrag_dpo',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    logging_steps=25,
    eval_steps=100,
    save_steps=200,
    fp16=True,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = DPOTrainer(
    model=model_dpo,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
)

print('Starting DPO training...')
print('Estimated time: ~8-15h on A100 80GB')
trainer.train()

# Save LoRA checkpoint
Path('checkpoints/meta_medrag_dpo').mkdir(parents=True, exist_ok=True)
model_dpo.save_pretrained('checkpoints/meta_medrag_dpo/lora_final')
tokenizer.save_pretrained('checkpoints/meta_medrag_dpo/lora_final')

import shutil
shutil.copytree('checkpoints/meta_medrag_dpo/lora_final',
                f'{SAVE_DIR}/dpo/lora_final', dirs_exist_ok=True)
print('DPO training complete — LoRA saved to Drive')

In [ ]:
# ── Cell 18: Save everything to Google Drive ────────────────────
import shutil, json
from pathlib import Path

files_to_save = [
    ('data/processed/activations_train_v2.pkl',
     f'{SAVE_DIR}/activations/activations_train_v2.pkl'),
    ('checkpoints/meco_probe_v2.pkl',
     f'{SAVE_DIR}/checkpoints/meco_probe_v2.pkl'),
    ('experiments/results/full_results.json',
     f'{SAVE_DIR}/results/full_results.json'),
    ('experiments/results/meco_score_distribution_v2.png',
     f'{SAVE_DIR}/results/meco_score_distribution_v2.png'),
    ('data/processed/dpo_pairs.json',
     f'{SAVE_DIR}/dpo/dpo_pairs.json'),
]

print('Saving all results to Google Drive...')
for src, dst in files_to_save:
    if Path(src).exists():
        shutil.copy(src, dst)
        print(f'  Saved: {dst}')
    else:
        print(f'  NOT FOUND: {src}')

# Download to local PC
from google.colab import files
for f_path in ['experiments/results/full_results.json',
               'checkpoints/meco_probe_v2.pkl',
               'experiments/results/meco_score_distribution_v2.png']:
    if Path(f_path).exists():
        files.download(f_path)

print('Done! All results saved.')